In [1]:
import pandas as pd

In [2]:
def combine_bedgraphs(plus_file, minus_file, output_file):
    columns = ["chrom", "start", "end", "value"]

    plus = pd.read_csv(plus_file, sep="\t", names=columns, comment="#")
    minus = pd.read_csv(minus_file, sep="\t", names=columns, comment="#")

    tracks = pd.concat([plus, minus], ignore_index=True)

    output = []

    for chrom, chrom_data in tracks.groupby("chrom", sort=False):
        events = {}

        for row in chrom_data.itertuples(index=False):
            events[row.start] = events.get(row.start, 0) + row.value
            events[row.end] = events.get(row.end, 0) - row.value

        current_value = 0
        previous_position = None

        for position in sorted(events):
            if previous_position is not None and position > previous_position and current_value != 0:
                output.append([chrom, previous_position, position, current_value])

            current_value += events[position]
            previous_position = position

    combined = pd.DataFrame(output, columns=columns)

    # Merge adjacent intervals with identical values.
    new_group = (
        (combined["chrom"] != combined["chrom"].shift()) |
        (combined["start"] != combined["end"].shift()) |
        (combined["value"] != combined["value"].shift())
    ).cumsum()

    combined = combined.groupby(new_group, as_index=False).agg(
        chrom=("chrom", "first"),
        start=("start", "first"),
        end=("end", "last"),
        value=("value", "first")
    )

    combined.to_csv(output_file, sep="\t", header=False, index=False)

    return combined

In [6]:
plus_file = "~/projects/RNA_protein_grammar/Stereogene_benchmark/TAU_m6A_bedgraphs/plus/1328_MAPTKD_WTTau_3XFLAG_IN_2.scaled.plus.bedGraph"
minus_file = "~/projects/RNA_protein_grammar/Stereogene_benchmark/TAU_m6A_bedgraphs/minus/1328_MAPTKD_WTTau_3XFLAG_IN_2.scaled.minus.bedGraph"
output_file = "~/projects/RNA_protein_grammar/Stereogene_benchmark/TAU_m6A_bedgraphs/1328_MAPTKD_WTTau_3XFLAG_IN_2.scaled.bedGraph"

combined = combine_bedgraphs(plus_file, minus_file, output_file)
combined.head()

/scratch/kflanagan/job_11886531/ipykernel_378575/2006780656.py:37: FutureWarning: A grouping was used that is not in the columns of the DataFrame and so was excluded from the result. This grouping will be included in a future version of pandas. Add the grouping as a column of the DataFrame to silence this warning.
  combined = combined.groupby(new_group, as_index=False).agg(


,chrom,start,end,value
0,chr1,14110,14178,0.087661
1,chr1,14442,14503,0.087661
2,chr1,14503,14504,0.175322
3,chr1,14504,14508,0.262983
4,chr1,14508,14515,0.175322
